# APA Metadata Tags Changelet Creation

This notebook generates `changelet.pb` files for APT Metadata addition scoped IDs.

For each scoped APT ID, the produced changelet adds the following metadata tags:

- `metadata:apa:improvement=yes`
- `metadata:apa:accuracy=rooftop`
- `metadata:apa:updated=<timestamp>`

In [ ]:
%scala
// Ticket: SEACO-6319
// List the re-scoped DC-Input CSV files (names ending with Already-On-BFP-DC-Input-format.csv)
// written by usa-already-on-bfp-apt-validation.ipynb. These files are read per-state in the next cell.
val reScopedFolder = "/mnt/aqua/data-correctness/correction-input/SEACO-6267-USA-Already-On-BFP-Scope/ReScoped_AlreadyOnBFP_Data/"
val reScopedFiles = dbutils.fs.ls(reScopedFolder)
  .filter(_.name.endsWith("Already-On-BFP-DC-Input-format.csv"))
  .map(_.path)

In [ ]:
%scala
// Ticket: SEACO-6319
// For every DC-Input CSV (one per US state), build a changelet that ADDs the three APA metadata
// tags to each APT row, then write one .pb per changelet under a per-state folder in
// ReScoped_AlreadyOnBfp_Changelets. Output path: {changeletFolder}/{State}/changelet_{State}_{randomId}.pb
// (State comes from the file name, e.g.
// AK_ALREADY_ON_CORRECT_BFP-Already-On-BFP-DC-Input-format-sample-run.csv -> AK).
import org.apache.spark.sql.functions._
import org.apache.spark.sql.{Dataset, Row}
import org.apache.spark.sql.Encoders.product
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types.IntegerType
import com.tomtom.orbis.Operation
import com.tomtom.orbis.io.spark.model.{Change, Changelet, ChangeletContext, Version, TagChange, SemanticChange, TagKey}
import com.tomtom.orbis.io.spark.changelet.mapper.ChangeletParser
import com.tomtom.addressing.bulk.scala.model.GroupedChanges
import java.io.{File, FileOutputStream}
import java.util.UUID
import scala.collection.mutable.ListBuffer

// --- Config -----------------------------------------------------------------
val LAYER_ID             = 14533
val CHANGELET_REVISION   = 41894243   // TODO: set to the layer 14533 revision these changelets target
val CHANGELET_BATCH_SIZE = 1000
// /dbfs path of the folder created under the scope directory (Java File IO needs the /dbfs prefix)
val changeletFolder = "/dbfs/mnt/aqua/data-correctness/correction-input/SEACO-6267-USA-Already-On-BFP-Scope/ReScoped_AlreadyOnBfp_Changelets"

// The three APA metadata tags every scoped APT gets; the timestamp is stamped once per run (epoch seconds).
val APA_UPDATED_TS = java.time.Instant.now().getEpochSecond.toString
val apaTags: Seq[(String, String)] = Seq(
  "metadata:apa:improvement" -> "yes",
  "metadata:apa:accuracy"    -> "rooftop",
  "metadata:apa:updated"     -> APA_UPDATED_TS
)

// Product_orbis_id in the DC-Input CSV is underscore-separated: {layerId}_{unsignedHigh}_{unsignedLow}
def createOrbisIdFromString(id: String): com.tomtom.orbis.io.spark.model.Id = {
  val s = id.split("_")
  new com.tomtom.orbis.io.spark.model.Id(
    layerId = Option(s(0).toInt),
    high = java.lang.Long.parseUnsignedLong(s(1)),
    low = java.lang.Long.parseUnsignedLong(s(2))
  )
}

// Read one DC-Input CSV, turn every row into an UPDATE Change carrying the three APA metadata tag
// additions, batch the changes into changelets, and write each changelet under a per-state folder.
def buildChangeletsForFile(path: String): (String, String, Long, Long) = {
  val fileName  = path.split("/").last
  val stateName = fileName.split("_")(0)   // e.g. AK_ALREADY_ON_CORRECT_BFP-...csv -> AK

  // Testing: skip the first 2 ids and read the next 2 (rows 3-4). Remove this window filter for a full run.
  val ids = spark.read.option("header", "true").csv(path)
    .select("Product_orbis_id")
    .filter(col("Product_orbis_id").isNotNull && col("Product_orbis_id") =!= "")
    .withColumn("_rn", row_number().over(Window.orderBy(lit(1))))
    .filter(col("_rn") > 2 && col("_rn") <= 4)
    .drop("_rn")

  // One Change per row: UPDATE the node, ADD the three plain metadata tags (no semantic changes).
  val changesDS: Dataset[Change] = ids.map { row =>
    val orbisId = createOrbisIdFromString(row.getAs[String]("Product_orbis_id"))
    val tagChange = ListBuffer[TagChange]()
    apaTags.foreach { case (k, v) =>
      tagChange += TagChange.apply(
        operation = Operation.ADD.getNumber.toByte,
        tagKey = TagKey.apply(k),
        value = Option(v),
        semanticIds = Option.empty,
        semanticChanges = Option.empty
      )
    }
    Change.nodeChange(id = orbisId, operation = Operation.UPDATE.getNumber.toByte, Option.empty, tagChange)
  }(product[Change])

  // Batch the changes (CHANGELET_BATCH_SIZE changes per changelet).
  val withBatchId = changesDS
    .withColumn("row_num", row_number().over(Window.orderBy(lit(1))))
    .withColumn("groupId", ((col("row_num") - 1) / CHANGELET_BATCH_SIZE).cast(IntegerType))
    .drop("row_num")

  val groupedChanges = withBatchId
    .groupBy("groupId")
    .agg(collect_list(struct("*")).alias("changes"))
    .select(col("groupId"), col("changes"))
    .as(product[GroupedChanges])

  val changeletDs: Dataset[Changelet] = groupedChanges.map { g =>
    Changelet(ChangeletContext(version = Version.apply(LAYER_ID, CHANGELET_REVISION)), g.changes)
  }(product)

  // Create the per-state output folder, then write each changelet into it.
  val stateFolder = s"$changeletFolder/$stateName"
  new File(stateFolder).mkdirs()

  val rowCount   = ids.count()
  val changelets = changeletDs.collect()
  changelets.foreach { changelet =>
    val randomSuffix = java.lang.Long.toUnsignedString(UUID.randomUUID().getLeastSignificantBits)
    val pbFilePath   = s"$stateFolder/changelet_${stateName}_${randomSuffix}.pb"
    val file = new File(pbFilePath)
    file.getParentFile.mkdirs()
    val out = new FileOutputStream(file)
    ChangeletParser.toChangelet(changelet).writeTo(out)
    out.close()
  }
  (fileName, stateName, rowCount, changelets.length.toLong)
}

// Testing: process only the FIRST DC-Input file. Switch back to reScopedFiles.map(...) for a full run.
val changeletResults = reScopedFiles.take(1).map(buildChangeletsForFile)

In [ ]:
%scala
changeletResults.foreach { case (fileName, state, rows, changelets) =>
  println(s"[$fileName] state=$state rows=$rows changelets=$changelets")
}
println(s"Total: files=${changeletResults.length}, " +
  s"rows=${changeletResults.map(_._3).sum}, " +
  s"changelets written=${changeletResults.map(_._4).sum} -> $changeletFolder")

In [ ]:
%scala
// Read a changelet .pb file back and print its content (protobuf text form) for verification.
import com.tomtom.orbis.Changelet
import java.io.FileInputStream

// >>> Set the .pb file to inspect (use the /dbfs path) <<<
val pbFilePathToRead = s"$changeletFolder/changelet_AK_REPLACE_WITH_RANDOM_ID.pb"

val in = new FileInputStream(pbFilePathToRead)
val changeletProto = try Changelet.parseFrom(in) finally in.close()

println(s"Reading changelet: $pbFilePathToRead")
println(changeletProto.toString)

In [ ]:
%scala
// Ticket: SEACO-6319
// Convert an unsigned 64-bit integer (as stored in Product_orbis_id) back to its signed Long form.
// java.lang.Long.parseUnsignedLong returns the signed Long whose bit pattern equals the unsigned
// value (values above Long.MAX_VALUE wrap to negative) - this is the signed representation used in
// signed-int id tables. Inverse of the signed->unsigned conversion in the source notebooks.
import org.apache.spark.sql.functions.udf

def unsignedToSignedLong(unsignedStr: String): Long = {
  if (unsignedStr == null || unsignedStr.isEmpty) 0L
  else try java.lang.Long.parseUnsignedLong(unsignedStr)
       catch { case _: NumberFormatException => unsignedStr.toLong }
}

// UDF form for use in DataFrame transformations.
val unsignedToSignedLongUDF = udf((unsignedStr: String) => unsignedToSignedLong(unsignedStr))

In [ ]:
%scala
// Ticket: SEACO-6319
// Accept a signed high/low pair and build the Product_orbis_id string "{layerId}_{high}_{low}",
// converting each signed Long to its unsigned string form (inverse of unsignedToSignedLong above).
import org.apache.spark.sql.functions.udf

def signedToUnsignedString(signedStr: String): String = {
  if (signedStr == null || signedStr.isEmpty) "0"
  else try java.lang.Long.toUnsignedString(signedStr.toLong)
       catch { case _: NumberFormatException => signedStr }
}

// {layerId}_{unsignedHigh}_{unsignedLow}, e.g. 14533_12345_67890
def buildIdHighLow(high: String, low: String): String =
  s"${LAYER_ID}_${signedToUnsignedString(high)}_${signedToUnsignedString(low)}"

// UDF form for use in DataFrame transformations.
val buildIdHighLowUDF = udf((high: String, low: String) => buildIdHighLow(high, low))

// Sample run (layer_id = LAYER_ID = 14533)
val sampleHigh = "11796342961399726080"
val sampleLow  = "7212821745216513619"
println(s"buildIdHighLow($sampleHigh, $sampleLow) = ${buildIdHighLow(sampleHigh, sampleLow)}")